# Multi-gNB scenario: two gNBs, three slices and configurable UEs

Run this notebook with the **Python (slices)** kernel. Run cells in order; deployment creates real resources. The notebook leaves the network running after traffic finishes.

- **Monolithic gNB:** `MONOLITHIC_UES` UEs on slice1, served by SMF1/UPF1.
- **One CU + two DUs:** `UES_PER_DU` UEs per DU; slices2/3 share SMF2/UPF2.
- **Gent VMs:** separate Kubernetes control plane, core/database, monitoring, gNB, CU, DU1, DU2, plus `UE_WORKERS` UE workers (7 + `UE_WORKERS` VMs).
- **Monitoring:** stock Open5GS, eBPF latency and N3 throughput, packet pairing, RAN metrics parsers and logs.
- **CSI:** the monolithic gNB and both DUs log SRS channel estimates per RB (`csi_per_rb.csv`), shown live on the monitoring VM and cut per traffic section by each experiment.

| UE group | SST / SD | DNN | Address pool | SMF / UPF |
|---|---|---|---|---|
| Monolithic, `MONOLITHIC_UES` UEs | 1 / absent | internet | 12.1.1.0/24 | smf1 / upf1 |
| DU1, `UES_PER_DU` UEs | 1 / 100000 | streaming | 14.1.1.0/24 | smf2 / upf2 |
| DU2, `UES_PER_DU` UEs | 1 / 200000 | ims | 16.1.1.0/24 | smf2 / upf2 |

Open Grafana after monitoring starts, attach the UEs, then run the traffic cells while displaying the dashboards. The default traffic takes about **80 seconds plus command/export overhead**, and can be rerun without redeployment.

The split gNB has a shared CU/N3 endpoint: the mapper identifies that CU, not individual DUs. The RAN parser metrics distinguish gNB/CU/DU services. SD values are hexadecimal; slice assignments do not impose bandwidth limits.

## Deployment overview

![Multi-gNB deployment topology](figures/deployment.png)


In [ ]:
import sys; print(sys.executable)

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
#%pip install --upgrade "git+https://github.com/yassir63/nador.git@main" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r "{REPOSITORY / 'ansible' / 'requirements.yaml'}"


Edit the project, infrastructure, VM flavor, UE counts and UE worker count here. `MONOLITHIC_UES` UEs attach to the monolithic gNB and `UES_PER_DU` UEs to each DU; they are spread over `UE_WORKERS` VMs.
RFsim runs radio and UEs in software: with too many UEs per VM the simulated radio falls behind real time (throughput, latency and CSI rates then no longer reflect a real cell). Start small and increase.
The experiment name includes a timestamp so this example does not reuse another run's experiment.
Keep the SLICES CLI authenticated in this environment.

In [ ]:
from datetime import UTC, datetime
import asyncio
import json

PROJECT = "post5g-beta"
EXPERIMENT = "multi-gnb-2du-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
INFRA = "be-gent1-bi-vm1"
VM_FLAVOR = "large"
MONOLITHIC_UES = 2  # UEs on the monolithic gNB
UES_PER_DU = 1  # UEs on each DU
UE_WORKERS = 1  # VMs running the UEs (node6, node7, ...)

SLICE_CONFIG = {
    "slice1": {"sst": 1, "sd": None, "dnn": "internet", "network": "12.1.1.0/24", "smf": "smf1", "upf": "upf1"},
    "slice2": {"sst": 1, "sd": "100000", "dnn": "streaming", "network": "14.1.1.0/24", "smf": "smf2", "upf": "upf2"},
    "slice3": {"sst": 1, "sd": "200000", "dnn": "ims", "network": "16.1.1.0/24", "smf": "smf2", "upf": "upf2"},
}
RADIO_SLICES = {"gnb": "slice1", "du1": "slice2", "du2": "slice3"}

UE_GROUPS = {
    "gnb": [f"mono_ue{i:02d}" for i in range(1, MONOLITHIC_UES + 1)],
    "du1": [f"du1_ue{i:02d}" for i in range(1, UES_PER_DU + 1)],
    "du2": [f"du2_ue{i:02d}" for i in range(1, UES_PER_DU + 1)],
}
CLIENTS = [client for group in UE_GROUPS.values() for client in group]
UE_RADIOS = {client: radio for radio, clients in UE_GROUPS.items() for client in clients}
UE_SLICES = {client: RADIO_SLICES[radio] for client, radio in UE_RADIOS.items()}

UE_NODES = [f"node{i}" for i in range(6, 6 + UE_WORKERS)]
print(f"Experiment: {EXPERIMENT}; {7 + UE_WORKERS} VMs; {len(CLIENTS)} UEs")

The metamodel defines two gNBs: monolithic gNB ID 1 and split gNB ID 2.
Both DUs point to the same CU and have separate DU IDs, cell IDs and RFsim RUs.
Every UE explicitly references its own `gnb`, `du1` or `du2` radio endpoint.

In [ ]:
from metamodel.models.metamodel import MetaModel

# Keep the shared core functions, replacing only the original SMF/UPF pair.
core_components = {
    name: component.model_dump(mode="json")
    for name, component in MetaModel().core.components.items()
    if component.type not in ("smf", "upf")
}
core_components.update(
    {
        "smf1": {"type": "smf", "config": {"snssaiList": ["slice1"], "upfs": ["upf1"]}},
        "upf1": {"type": "upf", "config": {"snssaiList": ["slice1"]}},
        "smf2": {"type": "smf", "config": {"snssaiList": ["slice2", "slice3"], "upfs": ["upf2"]}},
        "upf2": {"type": "upf", "config": {"snssaiList": ["slice2", "slice3"]}},
    }
)
for name in ("amf", "nrf"):
    core_components[name]["config"]["plmnList"] = [{"snssaiList": list(SLICE_CONFIG)}]
core_components["nssf"]["config"] = {
    "nsiInfoList": [
        {"snssai": name, "nsiInformationList": [{"nrfId": "http://open5gs-nrf:80", "nsiId": index + 1}]}
        for index, name in enumerate(SLICE_CONFIG)
    ],
    "taInfoList": [{"snssaiList": list(SLICE_CONFIG)}],
    "amfInfoList": [],
}

metamodel = MetaModel.model_validate(
    {
        "core": {"components": core_components},
        "slices": {
            name: {
                "sst": config["sst"],
                "sd": config["sd"],
                "dnns": {config["dnn"]: {"ipv4Network": config["network"]}},
            }
            for name, config in SLICE_CONFIG.items()
        },
        "ran": {
            "components": {
                "gnb": {
                    "type": "gnb",
                    "config": {"gnbId": 1, "nrCellId": 1, "physicalCellId": 0, "ru": "gnb_ru"},
                },
                "cu": {"type": "cu", "config": {"gnbId": 2, "nrCellId": 2}},
                "du1": {
                    "type": "du",
                    "config": {
                        "gnbId": 2,
                        "gnbDuId": 1,
                        "nrCellId": 2,
                        "physicalCellId": 1,
                        "cu_cp": "cu",
                        "cu_up": "cu",
                        "ru": "du1_ru",
                    },
                },
                "du2": {
                    "type": "du",
                    "config": {
                        "gnbId": 2,
                        "gnbDuId": 2,
                        "nrCellId": 3,
                        "physicalCellId": 2,
                        "cu_cp": "cu",
                        "cu_up": "cu",
                        "ru": "du2_ru",
                    },
                },
            },
        },
        "rus": {f"{radio}_ru": {"type": "simulated"} for radio in UE_GROUPS},
        "ues": {
            name: {
                "config": {
                    "supi": f"00101{100 + index:010d}",
                    "gnbs": [UE_RADIOS[name]],
                    "ru": f"{UE_RADIOS[name]}_ru",
                    "subscriptions": [{"snssai": UE_SLICES[name], "dnn": SLICE_CONFIG[UE_SLICES[name]]["dnn"]}],
                },
            }
            for index, name in enumerate(CLIENTS)
        },
    }
)

# Both gNBs advertise all three supported slices.
for component in metamodel.ran.components.values():
    component.config.plmnList[0].snssaiList = list(SLICE_CONFIG)


`node1` runs the Kubernetes control plane and iperf server.
`node10` runs the core and database; `node11` runs monitoring.
`node2` runs the monolithic gNB, `node3` the CU, `node4` DU1 and `node5` DU2.
UE workers start at `node6`; the UEs are distributed across those workers.

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "params": {
            "provisioning": {
                "project": PROJECT,
                "experiment": EXPERIMENT,
                "infra": INFRA,
                "flavor": VM_FLAVOR,
                "extra_path": "~/slices/bin",
                "create_experiment": True,
            },
            "k8s": {
                "master_node": "node1",
                "local_kubeconfig_path": str(Path(f".k8s-{EXPERIMENT}.conf").resolve()),
            },
        },
        "nodes": {
            "node1": {"roles": []},
            "node10": {"roles": ["core", "database"]},
            "node11": {"roles": []},
            **{name: {"roles": ["ran"]} for name in ("node2", "node3", "node4", "node5")},
            **{name: {"roles": ["ue"]} for name in UE_NODES},
        },
        "ran": {
            "implementation": "oai",
            "components": {
                "gnb": {"type": "gnb", "node_name": "node2"},
                "cu": {"type": "cu", "node_name": "node3"},
                "du1": {"type": "du", "node_name": "node4"},
                "du2": {"type": "du", "node_name": "node5"},
            },
        },
        "ues": {
            name: {"type": "containerized", "implementation": "oai", "node_name": UE_NODES[index % len(UE_NODES)]}
            for index, name in enumerate(CLIENTS)
        },
    }
)


In [ ]:
# TEMPORARY startup workarounds, isolated from installed upstream files.
# Retry a pod that disappears during readiness waiting; allow cold Helm image pulls.
import shutil
import tempfile
import yaml

local_roles = Path(tempfile.mkdtemp(prefix="multi-gnb-ansible-"))
for role in ("k8s/wait", "helm/install"):
    source = next(Path(path) / role for path in deployment_model.params.ansible.roles_paths
                  if (Path(path) / role).is_dir())
    destination = local_roles / role
    shutil.copytree(source, destination)
    tasks_file = destination / "tasks/main.yaml"
    tasks = yaml.safe_load(tasks_file.read_text())
    for task in tasks:
        info = task.get("kubernetes.core.k8s_info", {})
        if not info.get("wait"):
            continue
        if role == "k8s/wait" and info.get("kind") == "Pod":
            task.update(register="system_pods", until="system_pods is succeeded", retries=3, delay=5)
        elif role == "helm/install":
            info["wait_timeout"] = 600
    tasks_file.write_text(yaml.safe_dump(tasks, sort_keys=False))
deployment_model.params.ansible.roles_paths.insert(0, str(local_roles))


Create the VMs, then deploy Kubernetes. This is the first cell that provisions infrastructure.

In [ ]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(nodes=["node1"])
await prov.provision()
print(prov.get_ansible_inventory())


In [ ]:
k8s = K8SManagement(preceding=prov)
await k8s.deploy()


## Deploy the core

Use the stock upstream Open5GS images and templates. Advertise each SMF's assigned slices to NRF, then register every UE as a subscriber. Throughput comes from the eBPF probes; the old modified SMF/UPF metrics image and its KPI calculator are not used.

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=k8s)
for component in core.deployment_model.core.components.values():
    component.node_name = "node10"


In [ ]:
# Advertise only this SMF's slices/DNNs; keep upstream stock images and templates.
from orchestrator.models.core import ADVANCED_CORE_CONFIG_MAPPING

for name, component in core.deployment_model.core.components.items():
    if component.type != "smf":
        continue
    advanced = component.advanced_config.model_dump() if component.advanced_config else {}
    advanced["info"] = {"s_nssai": []}
    for slice_name in core.metamodel.core.components[name].config.snssaiList:
        snssai = core.metamodel.slices[slice_name]
        entry = {"sst": snssai.sst, "dnn": list(snssai.dnns)}
        if snssai.sd is not None:
            entry["sd"] = f"{snssai.sd:06x}"
        advanced["info"]["s_nssai"].append(entry)
    config_type = ADVANCED_CORE_CONFIG_MAPPING[component.implementation][component.type]
    component.advanced_config = config_type.model_validate(advanced)


In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## Deploy the radios

Start the monolithic gNB and CU, then DU1 and DU2. Local OAI charts include the metrics-parser sidecar on the gNB, CU and both DUs, and the CSI output volume on the gNB and both DUs.

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(preceding=core)


In [ ]:
# Select local OAI charts that include the metrics-parser sidecar.
from importlib.resources import files

oai_charts = files("nador") / "assets" / "ran" / "oai"
for name, component in ran.deployment_model.ran.components.items():
    if component.implementation == "oai" and component.type in ("gnb", "cu", "cu_cp", "cu_up", "du"):
        component.custom_chart_path = str(oai_charts / f"oai-{component.type}")
        component.custom_rendered_files = [
            str(oai_charts / "values.yaml.j2"),
            str(oai_charts / "config.yaml.j2"),
        ]


Enable CSI logging on the monolithic gNB and both DUs; the CU has no PHY, hence no SRS. `enable_csi` selects the
`r2labuser/oai-gnb-csi` image with the same OAI tag, forces periodic SRS and CSI-RS in the rendered gNB/DU
configuration, and mounts `/data/csi` from the node: each logger writes `/data/csi/csi_per_rb.csv` on its own VM
(`node2`, `node4`, `node5`), where the file survives pod restarts. Run it before preparing the radios. The other OAI
components, and all of them with an empty `CSI_COMPONENTS`, keep the image without CSI logger.

Choose the CSI mode in the next cell. `CSI_COMPONENTS` selects the radios that log (any subset of `gnb`, `du1`,
`du2`; an empty list disables CSI). `CSI_ENV` passes options to the logger, read when the radio starts, so a change
needs a radio redeployment:

| Option | Values | Default | Effect |
|---|---|---|---|
| `CSI_GRANULARITY` | `rb`, `subcarrier` | `rb` | One complex value per resource block, or per subcarrier (adds an `sc` column, 12× more rows) |
| `CSI_SUBCARRIER_SAMPLING` | 1 to 12 | 1 | In `subcarrier` mode, keep one subcarrier out of N |
| `CSI_ANTENNA_SELECTION` | `all` or indices, e.g. `0,1` | `all` | Receive antennas logged |
| `CSI_PORT_SELECTION` | `all` or indices, e.g. `0` | `all` | UE SRS ports logged |
| `CSI_FLUSH_CORE` | CPU number | 32 | CPU of the flush thread; on these VMs core 32 does not exist and the thread runs unpinned |

`CSI_COLLECT` sets how each experiment retrieves the CSI (see *CSI retrieval options* before the on-demand iperf).

In [ ]:
from nador.csi import CsiCollector, enable_csi

# CSI mode: radios that log (empty list: no CSI) and logger options (empty: defaults, RB granularity)
CSI_COMPONENTS = ["gnb", "du1", "du2"]
CSI_ENV = {}  # e.g. {"CSI_GRANULARITY": "subcarrier", "CSI_SUBCARRIER_SAMPLING": "2"}
# CSI retrieval by each ExperimentRun: whole file + per-section files, at the lowest priority on the radio VM
CSI_COLLECT = {"fetch_full": True, "window_types": "section", "padding_seconds": 0}

enable_csi(ran, components=CSI_COMPONENTS, env=CSI_ENV)
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

# One collector per CSI component, reused by every ExperimentRun below.
CSI_COLLECTORS = [CsiCollector(ran, name, **CSI_COLLECT) for name in CSI_COMPONENTS]

#### Default CSI settings

With the CSI cells as provided (`CSI_ENV = {}`), the settings below apply. Change them in the cells named in the last column; logger settings take effect when the radios start, so set them before preparing the radios.

| Setting | Value in this notebook | Where it is set |
|---|---|---|
| Granularity (`CSI_GRANULARITY`) | `rb`: one complex value per RB | `CSI_ENV` (logger default) |
| Subcarrier sampling (`CSI_SUBCARRIER_SAMPLING`) | 1 (only used with `subcarrier`) | `CSI_ENV` (logger default) |
| Antennas / ports (`CSI_ANTENNA_SELECTION` / `CSI_PORT_SELECTION`) | all | `CSI_ENV` (logger default) |
| Writer thread CPU (`CSI_FLUSH_CORE`) | 32 (left unpinned with a warning when the CPU does not exist) | `CSI_ENV` (logger default) |
| SRS period | chosen by OAI: smallest allowed period covering TDD period x maximum number of UEs per gNB (measured in the rfsim scenarios: 160 slots, 80 ms) | OAI gNB/DU (`do_SRS: periodic`) |
| CSI file | `/data/csi/csi_per_rb.csv` on the radio node (hostPath) | `enable_csi` (`output_dir`) |
| Whole CSI file copied per run (`fetch_full`) | yes | `CSI_COLLECT` |
| Windows cut (`window_types`, `padding_seconds`) | `section`, 0 s | `CSI_COLLECT` |
| Reading CPUs on the radio node (`cpu_list`) | any (lowest priority) | `CSI_COLLECT` |
| Temporary files on the radio node (`cleanup_remote`) | removed | `CsiCollector` default |
| Clock offset correction | applied only above 0.5 s (offset and drift recorded per run) | `ExperimentRun(csi_clock_threshold_s=0.5)` |
| Live view | on `node11`, ports `8501`, `8502`, `8503`, 120 s window, 10 s refresh, live copy restarted beyond 1024 MB, visualizer v9.0 (`oai-csi-logging` `csi-v3.1`) | `CSI_LIVE`, `CSI_LIVE_PORTS` |
| RFsim channel (`RFSIM_CHANNEL`) | `2x2+awgn`: 2 gNB/DU receive antennas, single-port SRS (see RFsim channel) | RFsim channel cell |

### RFsim channel

`RFSIM_CHANNEL` in the next cell selects the radio channel of the OAI gNBs/DUs with a simulated RU and of their UEs:

| `RFSIM_CHANNEL` | gNB/DU antennas | UE antennas (transmit / receive) | Channel model |
|---|---|---|---|
| `"2x2+awgn"` (default) | 2 (2 PUSCH ports, 2 MIMO layers) | 2 / 1 | AWGN, noise power -30 dB, no path loss |
| `"2x2+epa"` (still to be debugged) | 2 (2 PUSCH ports, 2 MIMO layers) | 2 / 1 | 3GPP EPA fading (7 paths up to 410 ns, 2x2 correlation) |
| `"awgn"` | 1 | 1 / 1 | the same AWGN model |
| `"ideal"` | 1 | 1 / 1 | none |

`"2x2+awgn"` is the radio configuration of the 5g_ansible rfsim scenario (`oai5g-rru` `gnb-config-rfsim.yaml` /
`du-config-rfsim.yaml`, UEs with `--ue-nb-ant-tx 2`). The OAI UE takes its antennas from the RU of its configuration,
so they are set on the simulated RU of each radio. Set it before preparing the radios and before the UE manager is
created.

What to expect in the CSI: OAI's AWGN channel is the identity matrix plus white noise (`random_channel.c`), and the
UEs send a single-port SRS (OAI UE capabilities without `--uecap_file`). In `"2x2+awgn"`, gNB/DU antenna 0 receives the
SRS and antenna 1 stays near 0 (it is linked to the second UE antenna, which sends no SRS). The estimates are flat
across RBs in every profile: OAI adds `10**(noise_power_dB/10) * 256` of noise per int16 sample, negligible at -30 dB
(0 dB already prevents UE attachment). `"2x2+epa"` uses a fading model instead: both gNB/DU antennas receive the SRS through different,
frequency-selective channels. RFsim draws the channel once per UE connection (OAI `simulator.cpp`), so the estimates
do not evolve over time for a connection, and the multipath convolution costs much more CPU than AWGN: check that the
radio VMs keep up (frame advance in the CSI, UE attachment).

Known issues with the OAI images used here (`develop-f80a883`, 2026-10-16), observed on SLICES BI:

- `"2x2+epa"` is still to be debugged. The EPA models load (`Model rfsimu_channel_ue0 type EPA allocated`), but the
  gNB/DU softmodem crashes as soon as a UE connects, right after `Random channel rfsimu_channel_ue0 in rfsimulator
  activated`: the kernel log of the radio VM shows `ru_thread: segfault at 0 ... error 4 in librfsimulator.so`, always
  at offset `0xf11b` (NULL pointer read in the rfsimulator when it applies the EPA channel). The UEs then lose their
  connection (`write() failed, errno(104)`) and no UE attaches. The configuration rendered by nador is the same
  as `"2x2+awgn"` except the channel type.
- `"2x2+awgn"`: the UEs attach (gNB/DU 2x2, UEs with 2 transmit and 1 receive antennas), but a UE softmodem may crash
  under sustained traffic (iperf): `DL__actor: segfault ... error 6 in libc.so.6`, always at offset `0x17173a`
  (invalid write while processing the downlink). Use `"awgn"` or `"ideal"` for long traffic experiments until it is
  understood.

In [ ]:
from nador.csi import set_rfsim_channel

# RFsim channel: "2x2+awgn" (default, as the 5g_ansible rfsim scenario), "2x2+epa" (fading, still to be debugged), "awgn" (1x1) or "ideal"
RFSIM_CHANNEL = "2x2+awgn"
set_rfsim_channel(ran, RFSIM_CHANNEL)
for name, component in ran.deployment_model.ran.components.items():
    advanced = component.advanced_config
    if advanced is not None and advanced.channelmod:
        antennas = (component.extra_env_vars or {}).get("RFSIM_ANTENNAS", "1")
        print(f"{name}: {antennas}x{antennas}, {advanced.channelmod['modellist_rfsimu_1'][0]['type']} channel")

In [ ]:
await ran.prepare(components=["gnb", "cu"])
await ran.install(components=["gnb", "cu"])
await ran.deploy(components=["du1"])
await ran.deploy(components=["du2"])


## Start monitoring before UE attachment

The AMF/SMF sniffers must be running before the UEs register. Latency probes observe the monolithic gNB, CU and both UPFs. Throughput probes observe both UPFs; RAN metrics are scraped from gNB/CU/DU parsers.

In [ ]:
from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")
monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)
await monitoring.deploy()


## Open Grafana

This tunnel uses the current VM address and SSH jump host from the deployment inventory. Open **N3 Throughput** for core traffic and slice/IMSI/PFCP-session mapping; use **Latency, RAN and resources** for latency, radio and CPU/memory panels. Credentials: **admin / admin**.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# Read the current Kubernetes control-plane connection from inventory.
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = [
    "ssh",
    "-N",
    "-T",
    "-S",
    "none",
    "-o",
    "ControlMaster=no",
    "-o",
    "ControlPersist=no",
    "-o",
    "BatchMode=yes",
    "-o",
    "ExitOnForwardFailure=yes",
    "-o",
    "ConnectTimeout=10",
]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += [
    "-p",
    str(ssh.port),
    "-L",
    f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
    f"{ssh.user}@{ssh.hostname or control}",
]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(
    f"[N3 Throughput]({GRAFANA_URL}/d/monitoring-n3-throughput?refresh=5s) · "
    f"[Latency, RAN and resources]({GRAFANA_URL}/d/afhb8p8jld91cb?refresh=5s) · "
    f"[Component logs]({GRAFANA_URL}/d/5g-component-logs)\n\n"
    "Username: **admin** · Password: **admin**"
))


## Open the live CSI views

One Streamlit view per CSI component runs on the monitoring VM (`node11`), never on the radio VMs: it follows the
radio VM's CSI file over SSH, where only `sshd` and `tail -F` run at the lowest priority. The views show a sliding
window (120 s by default) in local time and start before UE attachment, so SRS reports appear as each UE attaches.
The first start installs the viewer (about one minute); the tunnels use ports 8501–8503 on this machine.

`CSI_LIVE` sets the sliding window (`window_s`), the refresh period (`refresh_s`) and the size (`max_mb`) beyond which
the live copy on the monitoring VM restarts from "now"; `visualizer_src="<path>"` uses a local visualizer file
instead of the one downloaded from the `oai-csi-logging` repository. The live copy is for display only: experiments
always read the file on the radio VM. Stop the views with `await view.stop()` (see *Cleanup*).

In [ ]:
from nador.csi import CsiLiveView

# One live view per CSI component, all on the monitoring node, one port each
CSI_LIVE_PORTS = {"gnb": 8501, "du1": 8502, "du2": 8503}
CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024}
lives = {
    name: CsiLiveView(ran, live_node="node11", component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    for name in CSI_COMPONENTS
}
links = []
for name, view in lives.items():
    url = await view.start()
    links.append(f"[CSI {name}]({url})")
display(Markdown(" · ".join(links)))

## Attach the UEs

UEs deploy in batches of `UE_BATCH_SIZE` (four by default). Each batch finishes deployment before the next starts, limiting the RFsim startup burst. Each already has its own IMSI and slice subscription registered in the core.

In [ ]:
from orchestrator.deployment import UEManagement

ue = UEManagement(preceding=ran)
UE_BATCH_SIZE = 4

for start in range(0, len(CLIENTS), UE_BATCH_SIZE):
    batch = CLIENTS[start:start + UE_BATCH_SIZE]
    print(f"Deploying: {', '.join(batch)}")
    await asyncio.gather(
        *(ue.deploy(components=[client], quiet=True) for client in batch)
    )


Check that each CSI logger runs and writes its file: the gNB/DU logs show the logger start (the flush thread cannot
be pinned on these VMs, which is harmless), and each `csi_per_rb.csv` must grow between the two listings (the
logger flushes every 5 seconds).

In [ ]:
from orchestrator.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "[CSI]" in line or "Init: N_RB_DL" in line))

nodes = ProvisioningManagement(preceding=ue)
csi_nodes = [ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS]
for _ in range(2):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(command=["ls", "-l", "/data/csi/csi_per_rb.csv"], nodes=csi_nodes,
                                           become=True, quiet=True, flatten=False)
    for name, node in zip(CSI_COMPONENTS, csi_nodes):
        print(f"{name} ({node}): {listing[node].strip()}")
    await asyncio.sleep(10)

Read the assigned UE tunnel addresses and install iperf3 on the server VM.
Each UE must have a distinct address. This is also where you can inspect the two groups
before choosing traffic for your custom scenario.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = "node1"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5
UE_INTERFACE = "oaitun_ue1"

server_node = ue.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(ue.deployment_model.params.ansible.runner_path) / "iperf-server")


await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server VM",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "name": "Install iperf3",
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    },
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
addresses = await ue.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)

ue_ips = {}
for name in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[name]) for a in device["addr_info"] if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{name}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[name] = ipv4[0]

if len(set(ue_ips.values())) != len(CLIENTS):
    raise ValueError("Select UEs with different tunnel addresses")

if not QUERIES_FILE.is_file():
    raise FileNotFoundError(QUERIES_FILE)

print(f"iperf server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


Traffic helpers use the same UE-to-VM TCP commands as the passing RFsim examples.
Uplink sends from the UE; downlink adds iperf's reverse mode.

In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await ue.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    return json.loads(output[client])


## CSI retrieval options

Each `ExperimentRun(..., csi=CSI_COLLECTORS)` below collects the CSI when the run ends. On each radio VM, the
whole `csi_per_rb.csv` is read once at the lowest CPU/IO priority; only the results are copied here, under
`<run>/csi/<component>/`:

- `csi_per_rb.csv.gz`: the whole file, **since the radio started** (it grows during the deployment);
- `by_window/<level>/<window>/csi_per_rb.csv.gz`: the 5-second batches overlapping each window, with the file header;
- `csi_windows.json` (windows on the radio VM clock) and `csi_collection.json` (provenance and summary).

Options of `CSI_COLLECT` (set with the CSI mode, before the radios are prepared, or rebuild `CSI_COLLECTORS`):

| Option | Default | Effect |
|---|---|---|
| `fetch_full` | `True` | `False` copies only the per-window files: the whole file keeps growing, so repeated runs on one deployment copy more and more |
| `window_types` | `"section"` | Window levels cut from the file: `"section"`, `"campaign"` (whole run) or both (`"section,campaign"`) |
| `padding_seconds` | 0 | Margin added before and after each window |
| `cpu_list` | none | CPUs for the reading process on the radio VM (on R2Lab, housekeeping CPUs, never the isolated ones) |
| `cleanup_remote` | `True` | Remove the temporary work directory on the radio VM |

`ExperimentRun(..., csi_clock_threshold_s=0.5)` measures each radio VM clock against this machine and shifts the
windows only beyond this offset; drift during the run is reported in `experiment_metadata.json`.
To copy a CSI file outside an experiment (whole file only):
`await CsiCollector(ran, "gnb").collect("results/csi-gnb")`.

## On-demand iperf — choose a UE and direction

Run the iperf setup and helper cells above once, then use these two cells whenever you want traffic. Set `IPERF_UE` to a name from `UE_GROUPS`, choose `"uplink"` (UE → VM) or `"downlink"` (VM → UE), and set the duration in seconds. Rerun the configuration cell and execution cell for each test; deployment does not need to be repeated.

Examples: `mono_ue01` uses the monolithic gNB; `du1_ue01` and `du2_ue01` use the corresponding DUs. The run saves its iperf JSON report, configuration, timeline and Prometheus metrics under a new results directory, plus the CSI of the gNB and both DUs under `csi/<component>/`: the whole file since the radio started, and the part covering the iperf section.

In [ ]:
IPERF_UE = "du1_ue01"
IPERF_DIRECTION = "downlink"  # "uplink" or "downlink"
IPERF_SECONDS = 30

for radio, clients in UE_GROUPS.items():
    print(f"{radio}: {', '.join(clients)}")


In [ ]:
if IPERF_UE not in CLIENTS:
    raise ValueError(f"Unknown UE: {IPERF_UE}. Choose a name listed above.")
if IPERF_DIRECTION not in ("uplink", "downlink"):
    raise ValueError("Direction must be 'uplink' or 'downlink'.")
if IPERF_SECONDS <= 0:
    raise ValueError("Duration must be positive.")

iperf_port = BASE_PORT + 200 + CLIENTS.index(IPERF_UE)
iperf_directory = Path("results") / f"on-demand-{datetime.now(UTC):%Y%m%dT%H%M%S%fZ}"

print(f"Running {IPERF_UE}: {IPERF_DIRECTION}, {IPERF_SECONDS} seconds")
async with ExperimentRun(
    iperf_directory,
    name=f"{IPERF_UE}-{IPERF_DIRECTION}",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", {
        "client": IPERF_UE, "direction": IPERF_DIRECTION, "duration_seconds": IPERF_SECONDS,
        "ue_ip": ue_ips[IPERF_UE], "slice": UE_SLICES[IPERF_UE],
        "server_node": SERVER_NODE, "server_ip": server_ip, "port": iperf_port,
    })
    await start_iperf_server(iperf_port)
    with experiment.section("iperf"):
        iperf_report = await iperf_client(IPERF_UE, IPERF_DIRECTION, IPERF_SECONDS, iperf_port)
    experiment.save_json("iperf.json", iperf_report)
    # Allow the final traffic samples to reach Prometheus before artifact export.
    await asyncio.sleep(COLLECTION_SECONDS)

if iperf_report.get("error"):
    raise RuntimeError(iperf_report["error"])
print(f"Received: {iperf_report['end']['sum_received']['bits_per_second'] / 1e6:.2f} Mbit/s")
print(f"Artifacts: {iperf_directory.resolve()}")

Check the CSI collected by the last on-demand run. Each component reports its node clock (the offset to this
machine is only applied above 0.5 s; NTP keeps the BI VMs within a few milliseconds), the rows and 5-second batches
read, and the per-section files written under `csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((iperf_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("clock_offset_applied", "clock_offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(iperf_directory)) for p in (iperf_directory / "csi").rglob("*.gz")))

Optional: inspect the CSI of the iperf section for each component. Every UE of a radio appears, with or without
traffic, because SRS is periodic. In real time the frame number advances by about 500 per 5-second batch (modulo
1024); a much smaller advance means the RFsim radio runs slower than real time (overloaded VMs).

In [ ]:
import collections
import gzip

for name in CSI_COMPONENTS:
    path = iperf_directory / "csi" / name / "by_window" / "section" / "section__iperf" / "csi_per_rb.csv.gz"
    if not path.is_file():
        print(f"--- {name}: no CSI file for the iperf section")
        continue
    rntis, rows, first_frames, batch_start = collections.Counter(), 0, [], False
    with gzip.open(path, "rt") as f:
        for line in f:
            if line.startswith("# TIMESTAMP:"):
                batch_start = True
            elif line[:1].isdigit():
                rows += 1
                fields = line.split(",", 3)
                rntis[fields[2]] += 1
                if batch_start:
                    first_frames.append(int(fields[0]))
                    batch_start = False
    steps = [(b - a) % 1024 for a, b in zip(first_frames, first_frames[1:])]
    print(f"--- {name}: {len(first_frames)} batches, {rows} rows, RNTIs {dict(rntis)}")
    print(f"    frames per 5 s batch: {steps} (real time ~500, mod 1024)")

## Choose the demo traffic

All monolithic UEs run together, followed by all UEs of the two DUs. Each group runs **20 seconds uplink**, then **20 seconds downlink**. This produces four visible traffic batches and one JSON result per UE/direction.

Edit `TRAFFIC_CLIENTS` to use a subset, or change `DURATION` in the server setup cell. Rerun these traffic cells whenever you want another demo.

In [ ]:
TRAFFIC_CLIENTS = CLIENTS
TRAFFIC_GROUPS = {
    "monolithic": UE_GROUPS["gnb"],
    "split": UE_GROUPS["du1"] + UE_GROUPS["du2"],
}

SCENARIO = [
    {
        "name": f"{client}_{direction}",
        "batch": f"{group}_{direction}",
        "client": client,
        "direction": direction,
        "duration": DURATION,
    }
    for group, clients in TRAFFIC_GROUPS.items()
    for direction in ("uplink", "downlink")
    for client in clients
    if client in TRAFFIC_CLIENTS
]


## Run traffic and save artifacts

Keep Grafana open while this cell runs. `ExperimentRun` records the traffic timeline, Prometheus metrics, scenario and UE/slice assignments, plus each iperf JSON report and the CSI of the gNB and both DUs, cut per traffic batch under `csi/<component>/`. The baseline and collection windows allow scrapes before and after traffic. This is a normal experiment, with no trial runner or validation campaign.

In [ ]:
from itertools import groupby

timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="multi-gnb-2du",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)
    experiment.save_json("ue_groups.json", UE_GROUPS)
    experiment.save_json("slices.json", SLICE_CONFIG)
    experiment.save_json("ue_slices.json", UE_SLICES)
    experiment.save_json("ue_addresses.json", ue_ips)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    port = BASE_PORT
    for batch_name, grouped_steps in groupby(SCENARIO, key=lambda step: step["batch"]):
        steps = list(grouped_steps)
        ports = range(port, port + len(steps))
        print(f"Running {batch_name}: {len(steps)} client(s), {steps[0]['duration']} seconds")
        await asyncio.gather(*(start_iperf_server(server_port) for server_port in ports))

        with experiment.section(batch_name):
            reports = await asyncio.gather(
                *(
                    iperf_client(
                        client=test["client"],
                        direction=test["direction"],
                        duration=test["duration"],
                        port=server_port,
                    )
                    for test, server_port in zip(steps, ports)
                )
            )

        # Keep a report per UE, even when a client in this batch fails.
        for test, report in zip(steps, reports):
            experiment.save_json(f"{test['name']}.json", report)
        failures = {test["name"]: report["error"] for test, report in zip(steps, reports) if report.get("error")}
        if failures:
            raise RuntimeError(f"iperf failed: {failures}")
        total_mbps = sum(report["end"]["sum_received"]["bits_per_second"] for report in reports) / 1e6
        print(f"{batch_name}: {total_mbps:.2f} Mbit/s received in total")
        port += len(steps)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")

## Staggered traffic — add one UE every five seconds

Run this optional experiment after the iperf setup and helper cells. The default is the **DU1 UEs**, which share one DU. Choose `UE_GROUPS["du2"]`, `UE_GROUPS["gnb"]` or `CLIENTS` below for another group; UEs on different radios need not share capacity equally.

Each client starts five seconds after the previous one is scheduled. Earlier flows run longer, giving about 30 seconds with all clients active before they finish at approximately the same time. Remote command startup adds some timing variation. The artifacts record each client's command interval, and the iperf reports contain its traffic timing; the CSI is cut per client section.

Watch the per-session throughput curves in Grafana. With N clients the traffic lasts about `HOLD_SECONDS + (N - 1) × JOIN_DELAY` seconds, plus baseline, collection and startup overhead. This observes how sharing changes; it does not enforce equal rates.

In [ ]:
STAGGERED_CLIENTS = UE_GROUPS["du1"]
JOIN_DELAY = 5
HOLD_SECONDS = 30
STAGGERED_DIRECTION = "downlink"

staggered_scenario = [
    {
        "client": client,
        "start_after_seconds": index * JOIN_DELAY,
        "duration_seconds": HOLD_SECONDS + (len(STAGGERED_CLIENTS) - index - 1) * JOIN_DELAY,
        "port": BASE_PORT + 100 + index,
    }
    for index, client in enumerate(STAGGERED_CLIENTS)
]

# Start one flow after its delay, retaining a report and timeline for each UE.
async def staggered_flow(step, experiment):
    await asyncio.sleep(step["start_after_seconds"])
    await start_iperf_server(step["port"])
    client = step["client"]
    print(f"Starting {client}: {step['duration_seconds']} seconds {STAGGERED_DIRECTION}")
    with experiment.section(client):
        report = await iperf_client(
            client, STAGGERED_DIRECTION, step["duration_seconds"], step["port"]
        )
    experiment.save_json(f"{client}_{STAGGERED_DIRECTION}.json", report)
    if report.get("error"):
        raise RuntimeError(f"{client}: {report['error']}")

staggered_directory = Path("results") / f"staggered-{datetime.now(UTC):%Y%m%dT%H%M%S%fZ}"
async with ExperimentRun(
    staggered_directory,
    name="staggered-ue-traffic",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", {
        "direction": STAGGERED_DIRECTION,
        "ue_addresses": {client: ue_ips[client] for client in STAGGERED_CLIENTS},
        "ue_slices": {client: UE_SLICES[client] for client in STAGGERED_CLIENTS},
        "steps": staggered_scenario,
    })
    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)
    async with asyncio.TaskGroup() as traffic:
        for step in staggered_scenario:
            traffic.create_task(staggered_flow(step, experiment))
    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts: {staggered_directory.resolve()}")

In [ ]:
# Save both network and monitoring configuration for reference.
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")
monitoring.output(metamodel_file="monitoring_metamodel.yaml", deployment_model_file="monitoring_deployment_model.yaml")


### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before destroying the VMs: this closes the tunnels and revokes their SSH keys on the radio VMs.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

## Cleanup — only after the demonstration

Run the next cell **after** the demonstration (and after stopping the live CSI views) to destroy the VMs. To remove only monitoring, use `await monitoring.remove()` instead.

In [ ]:
await prov.unprovision()